# SF Case Files — Report Analysis

> *"The report is lying to you. I'm not sure how yet. But I will be."* — Adrian Monk

This notebook investigates the reports connected to the **SFCaseFiles** semantic model using Semantic Link Labs.

| Section | Theme | PBIR Required |
|---|---|---|
| Scene Survey | Inventory all reports and model connections | No |
| Cataloging the Evidence | Full ReportWrapper inspection | ✅ Yes |
| The Broken Evidence | Detect invalid field references | ✅ Yes |
| The Inspection | Report Best Practice Analyzer | ✅ Yes |
| The Field Report | Unused measures cross-reference | No |
| The Fix | Repair broken references + report actions | No / ✅ Yes |

---

⚠️ **PBIR Note:** Sections marked ✅ require reports to be in **PBIR format**.
PBIR is in preview but is becoming the default report format in Fabric — the future is already here.

---
## Setup

In [2]:
%run 00_Setup_Fresh

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 5, Finished, Available, Finished, True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fsspec-wrapper 0.1.15 requires PyJWT>=2.6.0, but you have pyjwt 2.4.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
✅ semantic-link-labs 0.14.0 installed
✅ Workspace    : FabConSQLCon2026
✅ Workspace ID : 21df5422-28b3-46aa-a1c5-ae2cd2a48578
✅ Lakehouse    : EvidenceLocker
✅ Lakehouse ID : a5f11fba-90dd-4722-81a9-d9cc0ebc6fdc
✅ Setup complete — the investigation begins


`ReportWrapper` is the primary interface for report inspection throughout this notebook.
`REPORT` must be in **PBIR format** for all sections marked ✅ in the table above.
`readonly=True` is the default — we investigate first, act later.

In [3]:
from sempy_labs.report import ReportWrapper

REPORT = "SFCaseFiles Report"  # must be PBIR format for sections marked ✅

# Connect to the report — used throughout this notebook
rpt = ReportWrapper(report=REPORT, workspace=WORKSPACE)

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 6, Finished, Available, Finished, False)

---
## Scene Survey — Inventory
> *"Before I touch anything, I need to know what's here."*
Catalog all reports in the workspace and confirm which semantic model each one is connected to.
A report pointing at the wrong model — or a deleted model — is already broken.
No PBIR required for this section — `fabric.list_reports()` works on any report format.

In [4]:
# All reports in the workspace
df_reports = fabric.list_reports(workspace=WORKSPACE)
display(df_reports)

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b5f87c87-7481-4f9f-9a96-dd0c4f9c4c7e)

In [5]:
# Which reports are connected to SFCaseFiles specifically?
df_sf_reports = df_reports[df_reports["Name"] == DATASET]
print(f"Reports connected to '{DATASET}': {len(df_sf_reports)}")
display(df_sf_reports)

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 8, Finished, Available, Finished, False)

Reports connected to 'SFCaseFiles': 1


SynapseWidget(Synapse.DataFrame, d202697d-1b41-4067-add7-850a1fd4aa11)

---
## Cataloging the Evidence — ReportWrapper Inventory
> *"Monk catalogs everything. Every page. Every visual. Every filter. Every bookmark."*
A full structural inventory of the report using `ReportWrapper`.
Each cell below answers a specific question about what's inside the report.
Requires PBIR format.

In [6]:
# Pages
print("=== Pages ===")
display(rpt.list_pages())

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 9, Finished, Available, Finished, False)

=== Pages ===


SynapseWidget(Synapse.DataFrame, f7c26f3b-37d8-4fb4-9c63-370a0694399e)

In [7]:
# Visuals — one row per visual across all pages
print("=== Visuals ===")
display(rpt.list_visuals())

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 10, Finished, Available, Finished, False)

=== Visuals ===


SynapseWidget(Synapse.DataFrame, 5b3578ea-e96a-470d-b246-567cc7a5ec0e)

In [8]:
# Visual objects — field-level breakdown of what each visual uses
print("=== Visual Objects ===")
display(rpt.list_visual_objects())

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 11, Finished, Available, Finished, False)

=== Visual Objects ===


SynapseWidget(Synapse.DataFrame, 204f808d-1bfd-4684-8653-c8f0e998ed26)

In [9]:
# Custom visuals — are there any third-party visuals installed?
# Unused custom visuals add overhead with no benefit
print("=== Custom Visuals ===")
display(rpt.list_custom_visuals())

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 12, Finished, Available, Finished, False)

=== Custom Visuals ===


SynapseWidget(Synapse.DataFrame, e6d9dc7b-6ca7-4ea6-b380-897cecf3ff5d)

In [10]:
# Bookmarks
print("=== Bookmarks ===")
display(rpt.list_bookmarks())

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 13, Finished, Available, Finished, False)

=== Bookmarks ===


SynapseWidget(Synapse.DataFrame, 4eb36c1d-95ad-4a4c-a370-005acc9ca70e)

In [11]:
# Visual interactions — which visuals cross-filter which?
print("=== Visual Interactions ===")
display(rpt.list_visual_interactions())

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 14, Finished, Available, Finished, False)

=== Visual Interactions ===


SynapseWidget(Synapse.DataFrame, 5fd5256a-25b4-455c-b00a-b1d1787ab1ec)

### Filter Audit
> *"Every filter is a constraint. Monk wants to know all the constraints."*
Filters exist at three tiers: report level (every page), page level (all visuals on a page),
and visual level (one visual only). Filters at the wrong tier are hard to find,
hard to maintain, and easy to forget — and nobody documents them.

In [12]:
# Report-level filters — apply to every page
print("=== Report Filters ===")
display(rpt.list_report_filters())

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 15, Finished, Available, Finished, False)

=== Report Filters ===


SynapseWidget(Synapse.DataFrame, fff43384-3258-46fc-9d7f-b2f018a4aa11)

In [13]:
# Page-level filters — apply to all visuals on a page
print("=== Page Filters ===")
display(rpt.list_page_filters())

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 16, Finished, Available, Finished, False)

=== Page Filters ===


SynapseWidget(Synapse.DataFrame, f9aebd2a-cfd3-47a2-91e8-b7d8f0d7520d)

In [14]:
# Visual-level filters — apply to a single visual only
print("=== Visual Filters ===")
display(rpt.list_visual_filters())

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 17, Finished, Available, Finished, False)

=== Visual Filters ===


SynapseWidget(Synapse.DataFrame, cc587a8d-b48c-45eb-a577-c5743ce12639)

### Report-Level Measures
> *"Monk doesn't let witnesses keep evidence at home. It belongs in the file."*
Report-level measures live inside the report definition, not the semantic model.
They can't be reused across reports, they're invisible to BPA, and they don't survive
a report rebuild. If you find any here — they belong in the model.

In [15]:
# Report-level measures — these should almost always live in the semantic model instead
print("=== Report-Level Measures ===")
df_rlm = rpt.list_report_level_measures()
display(df_rlm)

if df_rlm is not None and len(df_rlm) > 0:
    print(f"\n{len(df_rlm)} report-level measure(s) found.")
    print("   Consider migrating these to the semantic model for reuse and governance.")
else:
    print("No report-level measures found.")

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 18, Finished, Available, Finished, False)

=== Report-Level Measures ===


SynapseWidget(Synapse.DataFrame, 1fbd67b1-ed5b-48bd-b442-9b26a3f491c2)

No report-level measures found.


---

## ✅ The Broken Evidence — Invalid Visual References

> *"Something's not right. The visual is there. The field is gone. But nobody noticed."*

`list_semantic_model_objects(extended=True)` adds a `Valid Semantic Model Object` column.
`False` = that visual is referencing a measure, column, or hierarchy that no longer exists in the model.
This is a silent failure — the visual renders blank or errors, and users assume the data is missing.

In [16]:
# All semantic model objects used in this report — with validity check
df_smo = rpt.list_semantic_model_objects(extended=True)
display(df_smo)

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8dec7e01-3650-4414-87b9-59be26848425)

In [17]:
# Spotlight: just the broken ones
df_broken = df_smo[df_smo["Valid Semantic Model Object"] == False]

if df_broken.empty:
    print("✅ No broken visual references found. The report is clean.")
else:
    print(f"⚠️  {len(df_broken)} broken reference(s) found in '{REPORT}':")
    display(df_broken)

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 20, Finished, Available, Finished, False)

✅ No broken visual references found. The report is clean.


In [18]:
# Broken references across ALL reports connected to SFCaseFiles
# This is the org-wide version — no PBIR required
print("=== Broken references across all SFCaseFiles-connected reports ===")
df_all_smo = labs.list_report_semantic_model_objects(
    dataset=DATASET,
    workspace=WORKSPACE,
    extended=True
)
df_all_broken = df_all_smo[df_all_smo["Valid Semantic Model Object"] == False]

if df_all_broken.empty:
    print("✅ No broken references found across any connected reports.")
else:
    print(f"⚠️  {len(df_all_broken)} broken reference(s) found across all connected reports:")
    display(df_all_broken)

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 21, Finished, Available, Finished, False)

=== Broken references across all SFCaseFiles-connected reports ===
⚠️  2 broken reference(s) found across all connected reports:


SynapseWidget(Synapse.DataFrame, 769d94b5-77d3-46e5-b549-65a9614c3568)

---
## The Inspection — Report Best Practice Analyzer
> *"I have a list. I always have a list."*
The Report BPA evaluates the report structure against a set of rules covering
performance, error prevention, and formatting.
Start by viewing the default rules, then run the full inspection.
Requires PBIR format.

In [18]:
# View the default report BPA rules
df_rules = fabric.show_report_bpa_rules()
display(df_rules)

StatementMeta(, e482fb4d-0d83-4a64-9480-91dcd89daca2, 21, Finished, Available, Finished, False)

AttributeError: Attribute does not exist

In [19]:
# Run the Report BPA — HTML visualization
rep.run_report_bpa(report=REPORT, workspace=WORKSPACE)

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 22, Finished, Available, Finished, False)

Rule Name,Object Type,Object Name,Severity
"Reduce usage of filters on measuresMeasure filters may cause performance degradation, especially against a large semantic model.",Visual Filter,'First Page'[fe8509f67301d444a675] : 'fact_Incidents'[Total Incidents],⚠️


### Custom Rules
The default rules cover general report hygiene. You can also define your own rules as a DataFrame
and pass them in via the `rules` parameter.
The example below adds three targeted rules: one that explicitly catches broken semantic model
object references (the same check from The Broken Evidence section, but now enforced as a BPA rule),
plus two performance rules around visual count and visual object density.

In [20]:
# Run with custom rules — includes explicit broken reference check
import pandas as pd

custom_rules = pd.DataFrame(
    [
        (
            "Error Prevention",
            "Semantic Model",
            "Error",
            "Fix report objects which reference invalid semantic model objects",
            lambda df: df["Valid Semantic Model Object"] == False,
            "This rule highlights visuals, report filters, page filters or visual filters which reference an invalid semantic model object (i.e. Measure/Column/Hierarchy).",
            "",
        ),
        (
            "Performance",
            "Page",
            "Warning",
            "Reduce the number of visible visuals on the page",
            lambda df: df["Visible Visual Count"] > 15,
            "Reducing the number of visible visuals on a page will lead to faster report performance.",
            "",
        ),
        (
            "Performance",
            "Visual",
            "Warning",
            "Reduce the number of objects within visuals",
            lambda df: df["Visual Object Count"] > 5,
            "Reducing the number of objects (measures, columns) used in a visual will lead to faster report performance.",
            "",
        ),
    ],
    columns=["Category", "Scope", "Severity", "Rule Name", "Expression", "Description", "URL"],
)

rep.run_report_bpa(report=REPORT, workspace=WORKSPACE, rules=custom_rules)

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 23, Finished, Available, Finished, False)

### Persist the Results
Same pattern as the model BPA — `export=True` appends results to a delta table in the
attached lakehouse. Run it on a schedule alongside `run_model_bpa_bulk()` and you have
a full governance audit trail for both models and reports.

In [21]:
# Export BPA results to a delta table in the attached lakehouse
rep.run_report_bpa(report=REPORT, workspace=WORKSPACE, export=True)

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 24, Finished, Available, Finished, False)

🟢 The dataframe has been saved as the 'reportbparesults' table in the 'EvidenceLocker' lakehouse within the 'FabConSQLCon2026' workspace.


---
## The Field Report — What's Actually Used?
> *"You'd be amazed what people put in reports that nobody ever looks at."*
Cross-reference every measure in the semantic model against what the connected reports
actually reference. Unused measures are either dead weight, candidates for cleanup,
or — more interesting — evidence that something broke silently and nobody noticed.
No PBIR required for this section.

In [22]:
# All measures in the model
df_measures = fabric.list_measures(DATASET, workspace=WORKSPACE)

# All model objects used in reports connected to this model
df_used = labs.list_report_semantic_model_objects(
    dataset=DATASET,
    workspace=WORKSPACE,
    extended=True
)

used_measures = set(df_used[df_used["Object Type"] == "Measure"]["Object Name"].tolist())
all_measures  = set(df_measures["Measure Name"].tolist())
unused        = all_measures - used_measures

print(f"Total measures in model:       {len(all_measures)}")
print(f"Measures used in reports:      {len(used_measures)}")
print(f"Measures not used in reports:  {len(unused)}")

if unused:
    print("\nUnused measures:")
    for m in sorted(unused):
        print(f"  - {m}")

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 25, Finished, Available, Finished, False)

Total measures in model:       32
Measures used in reports:      5
Measures not used in reports:  27

Unused measures:
  - Active Crime Types
  - Active Neighborhoods
  - Avg Officers Per Incident
  - Case Burden Score
  - Case Chaos Index
  - Case Chaos Rating
  - Case Friction Score
  - Cold Case Rate
  - Downstream Panic Index
  - Geographic Scatter Score
  - High Severity Incidents
  - High Severity Rate
  - Incident Complexity Score
  - Investigation Noise Factor
  - Online Report Incidents
  - Online Report Rate
  - Open Case Burden Score
  - Operational Scatter Score
  - Severity Weighted Incidents
  - Slow Response Incidents
  - Slow Response Rate
  - Stale Incidents
  - Stale Rate
  - Uncleared Incidents
  - Uncleared Rate
  - Weekend Incidents
  - Weekend Rate


---
## The Fix — Actions and Repairs
> *"Now that I know what's wrong, I can fix it. I always fix it."*
Two categories of fixes:
1. **Repair** — programmatic field rename to resolve broken visual references
2. **Housekeeping** — report actions that clean up, standardize, and optimize the report
Everything in this section writes changes back to the service. Review before running.

### Repairing Broken Field References
# The pattern: pull the report JSON, find and replace the old field name, push it back.
Three steps, fully programmatic — no need to open the report in the browser and hunt through visuals.
Update `OLD_FIELD` and `NEW_FIELD` to match your actual rename before running.

In [23]:
# Step 1 — Pull the current report JSON
report_json = rep.get_report_json(report=REPORT, workspace=WORKSPACE)
print("Top-level keys:", list(report_json.keys()))

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, 26, Finished, Available, Finished, False)

ValueError: 🔴 Unable to retrieve report.json for the 'SFCaseFiles Report' report within the 'FabConSQLCon2026' workspace. This function only supports reports in the PBIR-Legacy format.

In [ ]:
# Step 2 — Find and replace a renamed field
import json

OLD_FIELD = "OldMeasureName"    # what the report currently references
NEW_FIELD = "New Measure Name"  # what the model field is now called

updated_json = json.loads(
    json.dumps(report_json).replace(OLD_FIELD, NEW_FIELD)
)
print(f"Replaced '{OLD_FIELD}' → '{NEW_FIELD}'")

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, -1, Cancelled, , Cancelled, True)

In [ ]:
# Step 3 — Push the updated JSON back
# ⚠️ Overwrites the report definition in the service
rep.update_report_from_reportjson(
    report=REPORT,
    report_json=updated_json,
    workspace=WORKSPACE
)
print(f"✅ '{REPORT}' updated.")

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, -1, Cancelled, , Cancelled, True)

### Report Housekeeping Actions

These functions write changes back to the service.
Uncomment and run as needed — all require PBIR format.

In [ ]:
# Apply a custom theme (from lakehouse or URL)
# rpt.set_theme(theme_file_path='/lakehouse/default/Files/SFCaseFiles_Theme.json')
# rpt.set_theme(theme_file_path='https://raw.githubusercontent.com/PowerBiDevCamp/FabricUserApiDemo/main/FabricUserApiDemo/DefinitionTemplates/Shared/Reports/StaticResources/SharedResources/BaseThemes/CY23SU08.json')

print("set_theme — uncomment to apply.")

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, -1, Cancelled, , Cancelled, True)

In [ ]:
# Hide/show a page
# rpt.set_page_visibility(page_name='Tooltip', hidden=True)

# Set the active page (first page shown when report opens)
# rpt.set_active_page(page_name='Overview')

print("Page visibility / active page — uncomment to apply.")

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, -1, Cancelled, , Cancelled, True)

In [ ]:
# Hide all tooltip and drillthrough pages (they shouldn't be visible in the nav)
# rpt.hide_tooltip_drillthrough_pages()

# Disable 'show items with no data' for all visuals (common performance drain)
# rpt.disable_show_items_with_no_data()

# Remove any custom visuals that aren't actually used in any visual on any page
# rpt.remove_unnecessary_custom_visuals()

print("Housekeeping actions — uncomment to apply.")

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, -1, Cancelled, , Cancelled, True)

In [ ]:
# Rebind report to a different semantic model
# Useful when a model is renamed, promoted, or replaced by a new version
# rep.report_rebind(
#     report=REPORT,
#     dataset="SFCaseFiles_V2",
#     report_workspace=WORKSPACE,
#     dataset_workspace=WORKSPACE
# )

print("report_rebind — uncomment to apply.")

StatementMeta(, 9344e940-5624-4aff-a4f6-0851f9502d26, -1, Cancelled, , Cancelled, True)

---

## Summary

| What | Function | PBIR Required |
|---|---|---|
| List all reports in workspace | `fabric.list_reports()` | No |
| Reports connected to a model | filter on `Dataset Name` | No |
| List pages | `rpt.list_pages()` | ✅ Yes |
| List visuals | `rpt.list_visuals()` | ✅ Yes |
| List visual objects (field-level) | `rpt.list_visual_objects()` | ✅ Yes |
| List custom visuals | `rpt.list_custom_visuals()` | ✅ Yes |
| List bookmarks | `rpt.list_bookmarks()` | ✅ Yes |
| List visual interactions | `rpt.list_visual_interactions()` | ✅ Yes |
| List report / page / visual filters | `rpt.list_report_filters()` etc. | ✅ Yes |
| Find report-level measures | `rpt.list_report_level_measures()` | ✅ Yes |
| Find broken visual references (single report) | `rpt.list_semantic_model_objects(extended=True)` | ✅ Yes |
| Find broken references (all reports) | `labs.list_report_semantic_model_objects(extended=True)` | No |
| Report Best Practice Analyzer | `rep.run_report_bpa()` | ✅ Yes |
| Unused measures analysis | `list_measures` + `list_report_semantic_model_objects` | No |
| Apply theme | `rpt.set_theme()` | ✅ Yes |
| Page visibility / active page | `rpt.set_page_visibility()` / `set_active_page()` | ✅ Yes |
| Remove unused custom visuals | `rpt.remove_unnecessary_custom_visuals()` | ✅ Yes |
| Hide tooltip/drillthrough pages | `rpt.hide_tooltip_drillthrough_pages()` | ✅ Yes |
| Disable show items with no data | `rpt.disable_show_items_with_no_data()` | ✅ Yes |
| Repair broken field references | `get_report_json` + replace + `update_report_from_reportjson` | No |
| Rebind report to new model | `rep.report_rebind()` | No |

> *"You'll thank me later. You always do."* — Adrian Monk

---

### Resources
- [Semantic Link Labs GitHub](https://github.com/microsoft/semantic-link-labs)
- [Report Analysis helper notebook](https://github.com/microsoft/semantic-link-labs/blob/main/notebooks/Report%20Analysis.ipynb)
- [sempy_labs.report docs](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.report.html)
- [thedaxshepherd.com](https://thedaxshepherd.com)
